In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale

In [2]:
def get_html(url = 'https://sindipetrocaxias.org.br/category/noticias/page/1'):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [13]:
def texto_para_datetime(texto: str) -> datetime:
    texto = texto.lower().replace("ago", "").strip()  # normaliza
    
    agora = datetime.now()
    
    if "segundo" in texto:
        qtd = int(texto.split()[0])
        return agora - timedelta(seconds=qtd)
    elif "minuto" in texto:
        qtd = int(texto.split()[0])
        return agora - timedelta(minutes=qtd)
    elif "hora" in texto:
        qtd = int(texto.split()[0])
        return agora - timedelta(hours=qtd)
    elif "dia" in texto:
        qtd = int(texto.split()[0])
        return agora - timedelta(days=qtd)
    else:
        raise ValueError(f"Formato não reconhecido: {texto}")

In [ ]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for i in range(len(articles) - 3):
        a = articles[i].find('a')
        link = a.get('href')
        span = articles[i].find('span', class_='post_meta_item post_date')
        date = span.text.strip()

        today = re.match(r".*ago$", date, re.IGNORECASE)

        if today:
            date = texto_para_datetime(date)
        else:
            date = datetime.strptime(date.lower(), "%d de %B de %Y")
            
        link_date = [link, date]
        news_links.append(link_date)

    return news_links

In [ ]:
def get_validated_links(news_links, min_date = datetime.datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [5]:
def get_next_page(lp_url, next_page_number = 1):
    validated_news_links = []
    lp_url = 'https://sindipetrocaxias.org.br/category/noticias/page/'
    url = lp_url + str(next_page_number)
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(lp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [6]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    divs = soup.find('div', class_='post_content post_content_single entry-content')

    if divs:
        paragraphs = divs.find_all('p')
    else:
        paragraphs = []

    return title, paragraphs

In [7]:
def main():
    caxias_url = 'https://sindipetrocaxias.org.br/category/noticias/page/'
    next_page_number = 1
    validated_news_links = []
    url = caxias_url + str(next_page_number)
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(caxias_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'Caxias',
                    'url' : url,
                    'titulo' : title,
                    'data': str(validated_news_links[0][1]).split(' ')[0],
                    'paragrafo' : paragraph.text,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [15]:
result = main()

df = pd.DataFrame(result)

df

c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sindipetrocaxias.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sindipetrocaxias.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sindipetrocaxias.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-

,sindicato,url,titulo,data,paragrafo,num_paragrafo
0,Caxias,https://sindipetrocaxias.org.br/sindipetro-par...,Sindipetro participa de apuração sobre inciden...,2025-08-18,No dia 13/07 por volta de 2:45h um brigadista ...,1
1,Caxias,https://sindipetrocaxias.org.br/sindipetro-par...,Sindipetro participa de apuração sobre inciden...,2025-08-18,Foi constituída uma comissão de análise para a...,2
2,Caxias,https://sindipetrocaxias.org.br/nota-de-pesar-...,Nota de pesar pelo falecimento do companheiro ...,2025-08-18,O Sindipetro Caxias lamenta o falecimento do c...,1
3,Caxias,https://sindipetrocaxias.org.br/nota-de-pesar-...,Nota de pesar pelo falecimento do companheiro ...,2025-08-18,"O velório ocorrerá no Dia 17/08, as 14:00, no ...",2
4,Caxias,https://sindipetrocaxias.org.br/nota-de-pesar-...,Nota de pesar pelo falecimento do companheiro ...,2025-08-18,Deixamos aqui o nosso mais profundo sentimento...,3
...,...,...,...,...,...,...
82,Caxias,https://sindipetrocaxias.org.br/convocacao-de-...,Convocação de Assembleia – Rocha/TESIP (08/08) 7h,2025-08-18,EDITAL ASSEMBLEIA GERAL EXTRAORDINÁRIA,1
83,Caxias,https://sindipetrocaxias.org.br/convocacao-de-...,Convocação de Assembleia – Rocha/TESIP (08/08) 7h,2025-08-18,"Pelo presente edital, conforme artigo 12 parág...",2
84,Caxias,https://sindipetrocaxias.org.br/convocacao-de-...,Convocação de Assembleia – Rocha/TESIP (08/08) 7h,2025-08-18,1) ACT 2024.,3
85,Caxias,https://sindipetrocaxias.org.br/convocacao-de-...,Convocação de Assembleia – Rocha/TESIP (08/08) 7h,2025-08-18,"Duque de Caxias, 04 de agosto de 2025",4


In [16]:
df2 = df.explode('paragrafo')
df2.to_dict('records')

[{'sindicato': 'Caxias',
  'url': 'https://sindipetrocaxias.org.br/sindipetro-participa-de-apuracao-sobre-incidente-ocorrido-na-termorio/',
  'titulo': 'Sindipetro participa de apuração sobre incidente ocorrido na TERMORIO',
  'data': '2025-08-18',
  'paragrafo': 'No dia 13/07 por volta de 2:45h um brigadista civil e um operador da UTE realizavam uma manobra para abertura de um hidrante, quando ocorreu a quebra da conexão rosqueada, projetando o conjunto fixo da válvula. O jato d’água atingiu o brigadista, que sofreu queda de mesma altura no solo britado. Felizmente o colega nada sofreu e se encontra bem.',
  'num_paragrafo': 1},
 {'sindicato': 'Caxias',
  'url': 'https://sindipetrocaxias.org.br/sindipetro-participa-de-apuracao-sobre-incidente-ocorrido-na-termorio/',
  'titulo': 'Sindipetro participa de apuração sobre incidente ocorrido na TERMORIO',
  'data': '2025-08-18',
  'paragrafo': 'Foi constituída uma comissão de análise para a investigação do ocorrido, com a participação do Si